In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import skew

In [12]:
from google.colab import drive
drive.mount('/content/drive')
!ls -l "/content/drive/MyDrive/alrIEEEna_26_dataset/MLChallengeDataset/"

train = pd.read_csv("/content/drive/MyDrive/alrIEEEna_26_dataset/MLChallengeDataset/TRAIN.csv")
test = pd.read_csv("/content/drive/MyDrive/alrIEEEna_26_dataset/MLChallengeDataset/TEST.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 22878
-rw------- 1 root root     2076 Mar  3 08:35 readme.txt
-rw------- 1 root root  4711205 Mar  3 08:35 TEST.csv
-rw------- 1 root root 18712865 Mar  3 08:35 TRAIN.csv


In [13]:
X = train.drop("Class", axis=1)
y = train["Class"]
X_test = test.drop("ID", axis=1)
test_ids = test["ID"]

In [14]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

In [15]:
for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.01, num_leaves=31)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50)])

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / n_splits

[LightGBM] [Info] Number of positive: 13848, number of negative: 21172
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.021689 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11985
[LightGBM] [Info] Number of data points in the train set: 35020, number of used features: 47
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.395431 -> initscore=-0.424539
[LightGBM] [Info] Start training from score -0.424539
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.109857
[LightGBM] [Info] Number of positive: 13849, number of negative: 21172
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035584 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11985
[LightGBM] [Info] Number of data points in the train set: 35021, number of used featur

In [17]:
from sklearn.metrics import accuracy_score

best_thresh = 0.5
best_score = 0
for t in np.linspace(0.3, 0.7, 41):
    score = accuracy_score(y, (oof_preds > t).astype(int))
    if score > best_score:
        best_score = score
        best_thresh = t

In [18]:
final_labels = (test_preds > best_thresh).astype(int)
submission = pd.DataFrame({"ID": test_ids, "Class": final_labels})
submission.to_csv("submission.csv", index=False)

In [19]:
from sklearn.metrics import f1_score

binary_predictions = (oof_preds > best_thresh).astype(int)


f1 = f1_score(y, binary_predictions)
print(f"The F1 score of the model is: {f1}")

The F1 score of the model is: 0.9664767303965522
